In [1]:
!pip install faker pandas

import pandas as pd
from faker import Faker
import random
from datetime import datetime, timedelta

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 23.7 MB/s eta 0:00:00


In [2]:
# Initialize Faker
fake = Faker()

# Define parameters
NUM_TWEETS = 20000
start_date = datetime.now() - timedelta(days=30)

In [3]:
# Real-world disaster/weather event configurations
event_templates = {
    "disaster": [
        ("earthquake", "Magnitude {magnitude} earthquake felt in {location}. Stay safe everyone."),
        ("flood", "The river is overflowing! Massive flood hitting downtown {location}."),
        ("wildfire", "Wildfire spreading rapidly near {location}. Evacuation orders in place."),
        ("terror attack", "Breaking: Reports of a terror attack near the central station in {location}."),
        ("explosion", "Huge explosion reported at an industrial plant in {location}.")
    ],
    "incident": [
        ("accident", "Major traffic accident on the main highway outside {location}. Expect delays."),
        ("power outage", "Total power outage across {location} grid. Everything is black."),
        ("train derailment", "Emergency services responding to a train derailment near {location}."),
        ("structural collapse", "Building structural collapse reported in {location}. First responders on scene."),
        ("cyber attack", "Local government systems in {location} hit by a massive cyber attack.")
    ],
    "weather": [
        ("blizzard", "Heavy snow and blizzard conditions blinding drivers in {location}."),
        ("hurricane", "Category 3 hurricane making landfall near {location} tonight."),
        ("tornado", "Tornado warning issued for {location} and surrounding counties! Take cover!"),
        ("heatwave", "Record-breaking heatwave in {location}. Temperatures reaching 45C."),
        ("drought", "Severe drought causing water rationing across the entire {location} region.")
    ]
}

In [4]:
famous_locations = [
    "New York, USA", "London, UK", "Tokyo, Japan", "Paris, France",
    "Sydney, Australia", "Mumbai, India", "Cairo, Egypt", "Rio de Janeiro, Brazil",
    "Cape Town, South Africa", "Toronto, Canada", "Berlin, Germany", "Beijing, China"
]


In [5]:
# Pools for noise generation
emojis_pool = ["🚨", "⚠️", "📢", "🛑", "🌧️", "🔥", "💨", "❄️", "💥", "😱", "🙏", "👇", "💔"]
symbols_pool = ["!!!", "??", "##", "**CRITICAL**", "[URGENT]", "->", "=>", "@USER"]
url_domains = ["https://bit.ly", "https://t.co", "https://news.org"]

data = []

for _ in range(NUM_TWEETS):
    # 1. Setup core information
    target = random.choice(["disaster", "incident", "weather"])
    keyword, template = random.choice(event_templates[target])
    text_location = random.choice(famous_locations)
    magnitude = round(random.uniform(4.0, 7.9), 1)

    # 2. Build base tweet text
    text = template.format(location=text_location, magnitude=magnitude)

    # 3. Inject URL (80% chance)
    if random.random() > 0.2:
        fake_url = f"{random.choice(url_domains)}{fake.random_number(digits=7, fix_len=True)}"
        text += f" {fake_url}"

    # 4. Inject random emojis (70% chance)
    if random.random() > 0.3:
        selected_emojis = "".join(random.choices(emojis_pool, k=random.randint(1, 3)))
        text = f"{selected_emojis} {text}"

    # 5. Inject random symbols or punctuation noise (60% chance)
    if random.random() > 0.4:
        noise = random.choice(symbols_pool)
        if random.choice([True, False]):
            text = f"{noise} {text}"
        else:
            text = f"{text} {noise}"

    # 6. Randomize Null values for structural columns (20% chance)
    row_location = text_location if random.random() > 0.2 else None
    row_keyword = keyword if random.random() > 0.2 else None

    data.append({
        "id": (_+1),
        "keyword": row_keyword,
        "location": row_location,
        "text": text,
        "target": target,
        "created_at": fake.date_time_between_dates(datetime_start=start_date, datetime_end=datetime.now()).strftime('%Y-%m-%d %H:%M:%S'),
    })

# Convert to DataFrame and save
df = pd.DataFrame(data)
df.to_csv('./train.csv', index=False)

print(f"Successfully generated {NUM_TWEETS} noisy tweets and saved to 'train.csv'")
print("\nSample Data Preview:")
pd.set_option('display.max_colwidth', None)
print(df[['id','keyword', 'target', 'text','location']].head())


Successfully generated 20000 noisy tweets and saved to 'train.csv'

Sample Data Preview:
   id              keyword    target  \
0   1             blizzard   weather   
1   2                 None  incident   
2   3        terror attack  disaster   
3   4  structural collapse  incident   
4   5              drought   weather   

                                                                                                               text  \
0                        Heavy snow and blizzard conditions blinding drivers in Cairo, Egypt. https://bit.ly4336140   
1       🔥🚨 Emergency services responding to a train derailment near Rio de Janeiro, Brazil. https://news.org8898222   
2         Breaking: Reports of a terror attack near the central station in Berlin, Germany. https://news.org1639884   
3  [URGENT] 🙏 Building structural collapse reported in Cairo, Egypt. First responders on scene. https://t.co2740413   
4     💔 Severe drought causing water rationing across the entire Beijing, C